# 08 · Noticias y bolsa

**Objetivo:** medir si el sentimiento de las noticias de una empresa se relaciona con lo que hace su acción después. Los tres modelos del notebook 07 (M1 diccionario, M2 Transformer y M3 BERT) puntúan los titulares de FNSPID que mencionan a su empresa, y se compara la rentabilidad de la acción, menos la del mercado, después de las noticias positivas, neutrales y negativas. Antes se comprueba con 200 titulares recientes etiquetados a mano si los modelos, entrenados con frases de PhraseBank, también saben leer titulares.

**Plan:**

| Paso | Qué se hace | Estado |
|---|---|---|
| 1 | Plantilla de 200 titulares de NewsAPI para etiquetar a mano, con la regla de etiquetado | hecho |
| 2 | Las capas y el vocabulario de M2 y M3, desde `funciones.py` | pendiente |
| 3 | M1, M2 y M3 puntúan los titulares de FNSPID que mencionan a su empresa | pendiente |
| 4 | Los tres modelos con los 200 titulares etiquetados a mano | pendiente |
| 5 | SPY y rentabilidad de cada acción menos la del mercado | pendiente |
| 6 | Sentimiento diario por empresa y estudio de eventos | pendiente |

**Entradas:**
- `data/cache/noticias_newsapi.parquet` y `noticias_fnspid.parquet` (notebook 06)
- `data/cache/empresas_sec.parquet` (notebook 03): nombre de cada empresa
- `etiquetas/titulares_a_mano.csv`: los 200 titulares, etiquetados a mano por el autor

**Salidas:**
- `etiquetas/titulares_a_mano.csv`: la plantilla (solo si no existe; nunca se sobrescribe)

In [1]:
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

from funciones import (
    CARPETA_CACHE,
    CLASES_SENTIMIENTO,
    SEMILLA,
)

load_dotenv();

## 1. ¿Saben los modelos leer titulares? La plantilla para etiquetar a mano

Los modelos del 07 se entrenaron y se examinaron con frases de **notas de prensa de empresas finlandesas de 2004 a 2010** (Financial PhraseBank). Aquí se aplican a **titulares de noticias de empresas de EEUU**, más cortos y con otra jerga. Si en este notebook el sentimiento no se relacionara con la bolsa, habría que saber si es porque no hay relación o porque los modelos no saben leer titulares. No existe un conjunto de titulares de nuestras empresas con su sentimiento etiquetado, así que el autor etiqueta **200 a mano**.

**Qué titulares:** los recientes de NewsAPI (notebook 06), que ya mencionan a su empresa. Son de otra época que los de FNSPID, así que es una prueba **fuera de dominio** en dos sentidos: otro tipo de texto que PhraseBank y otras fechas. Se eligen al azar con la semilla del proyecto, sin titulares repetidos y **como mucho 5 por empresa**, para que unas pocas empresas muy citadas (Amazon tiene el 14 % de los titulares) no se lleven la muestra. Para que la muestra sea siempre la misma aunque se vuelva a ejecutar el 06, se usan solo los titulares descargados hasta el 16 nov 2026, el día en que se hizo.

**La regla de etiquetado** se escribió antes de empezar, en `etiquetas/regla_de_etiquetado.md`. Es la misma pregunta que se hizo a los anotadores de PhraseBank: desde el punto de vista de un inversor en esa empresa, ¿el titular es una noticia buena (`positive`), mala (`negative`) o neutra (`neutral`) para el precio de su acción? Solo con el titular, que es lo que ven los modelos; en la duda, `neutral`; y sin mirar lo que dicen los modelos.

**Titulares que no tratan de la empresa.** El filtro del 06 (`menciona_empresa`) acepta un titular si contiene el ticker o el nombre de la empresa, y a veces coinciden con una palabra corriente (*principal*, *hinge*, *holding*, *SNAP*). Esos titulares no se quitan: también están entre los de FNSPID, así que forman parte de lo que leerán los modelos. Se marcan en una columna, `trata_de_la_empresa` (`sí` o `no`), y se etiquetan como `neutral`, porque no deberían mover la acción. Así se pueden dar dos resultados: con los 200 titulares (lo que pasará en FNSPID) y solo con los que tratan de la empresa (cómo leen los modelos un titular de verdad). Y se mide de paso cuántas veces se equivoca el filtro.

In [2]:
CARPETA_ETIQUETAS = Path("etiquetas")
RUTA_ETIQUETADO = CARPETA_ETIQUETAS / "titulares_a_mano.csv"
TITULARES_A_MANO = 200
MAXIMO_POR_EMPRESA = 5
FECHA_MUESTRA = pd.Timestamp("2026-11-16")   # titulares descargados hasta el día en que se hizo la muestra

noticias_recientes = pd.read_parquet(CARPETA_CACHE / "noticias_newsapi.parquet")
noticias_recientes = noticias_recientes[noticias_recientes["descargado"] <= FECHA_MUESTRA]
print(f"Titulares de NewsAPI: {len(noticias_recientes):,} · distintos: {noticias_recientes['titular'].nunique():,}"
      f" · empresas: {noticias_recientes['ticker'].nunique()}"
      f" · del {noticias_recientes['fecha'].min():%d %b %Y} al {noticias_recientes['fecha'].max():%d %b %Y}")

Titulares de NewsAPI: 3,036 · distintos: 2,671 · empresas: 106 · del 20 Sep 2026 al 15 Nov 2026


La muestra: se barajan los titulares distintos con la semilla, se quedan los 5 primeros de cada empresa y, de ellos, se eligen 200 al azar. Se añade el nombre de la empresa para saber de cuál se juzga cada titular.

In [3]:
nombres = pd.read_parquet(CARPETA_CACHE / "empresas_sec.parquet").set_index("ticker")["name"]

candidatos = (noticias_recientes.drop_duplicates("titular")
              .sample(frac=1, random_state=SEMILLA)          # orden al azar, con semilla
              .groupby("ticker").head(MAXIMO_POR_EMPRESA))   # como mucho 5 por empresa
muestra = candidatos.sample(TITULARES_A_MANO, random_state=SEMILLA)

plantilla = pd.DataFrame({
    "id": range(1, TITULARES_A_MANO + 1),
    "ticker": muestra["ticker"].to_numpy(),
    "empresa": muestra["ticker"].map(nombres).to_numpy(),
    "titular": muestra["titular"].to_numpy(),
    "trata_de_la_empresa": "",
    "etiqueta": "",
})
print(f"Candidatos (como mucho {MAXIMO_POR_EMPRESA} por empresa): {len(candidatos)} · elegidos: {len(plantilla)}"
      f" · empresas en la muestra: {plantilla['ticker'].nunique()}")
plantilla.head()

Candidatos (como mucho 5 por empresa): 457 · elegidos: 200 · empresas en la muestra: 93


,id,ticker,empresa,titular,trata_de_la_empresa,etiqueta
0,1,KMI,"KINDER MORGAN, INC.",Energy Transfer vs. Kinder Morgan: Which Natur...,,
1,2,PFG,PRINCIPAL FINANCIAL GROUP INC,School principal documents Indonesia's worst w...,,
2,3,DDOG,"Datadog, Inc.",Here Are Monday’s Top Wall Street Analyst Rese...,,
3,4,TMUS,"T-Mobile US, Inc.",Do You Like Cash? Consider T-Mobile Stock,,
4,5,PENG,"Penguin Solutions, Inc.",Penguin Solutions Trumpets AI-Driven Beat-And-...,,


La plantilla se escribe **solo si no existe**: una vez que el autor empieza a etiquetar, volver a ejecutar el notebook nunca borra sus etiquetas.

In [4]:
if RUTA_ETIQUETADO.exists():
    print(f"{RUTA_ETIQUETADO} ya existe: no se sobrescribe")
else:
    CARPETA_ETIQUETAS.mkdir(exist_ok=True)
    plantilla.to_csv(RUTA_ETIQUETADO, index=False, encoding="utf-8")
    print(f"Plantilla escrita: {RUTA_ETIQUETADO} ({len(plantilla)} titulares)")

Plantilla escrita: etiquetas/titulares_a_mano.csv (200 titulares)


Cuántos titulares hay etiquetados. El fichero se lee de forma flexible (con comas o con punto y coma, que es como lo guardan algunas hojas de cálculo) y sin tener en cuenta mayúsculas ni espacios (*si* vale como *sí*). Avisa de las filas a medias o mal escritas y de las que no tratan de la empresa pero no son `neutral`.

In [5]:
etiquetado = pd.read_csv(RUTA_ETIQUETADO, sep=None, engine="python", keep_default_na=False)
etiquetado["etiqueta"] = etiquetado["etiqueta"].str.strip().str.lower()
etiquetado["trata_de_la_empresa"] = etiquetado["trata_de_la_empresa"].str.strip().str.lower().replace({"si": "sí"})
hechos = etiquetado["etiqueta"].isin(CLASES_SENTIMIENTO) & etiquetado["trata_de_la_empresa"].isin(["sí", "no"])
empezados = (etiquetado["etiqueta"] != "") | (etiquetado["trata_de_la_empresa"] != "")
mal_escritos = etiquetado[empezados & ~hechos]
no_neutrales_ajenos = etiquetado[(etiquetado["trata_de_la_empresa"] == "no") & (etiquetado["etiqueta"] != "neutral")]

print(f"Etiquetados: {hechos.sum()} de {len(etiquetado)}")
print("Reparto:", etiquetado.loc[hechos, "etiqueta"].value_counts().to_dict(),
      "· no tratan de la empresa:", (etiquetado.loc[hechos, "trata_de_la_empresa"] == "no").sum())
if len(mal_escritos):
    print("A medias o mal escritos (etiqueta: positive, negative o neutral; trata_de_la_empresa: sí o no):")
    display(mal_escritos[["id", "titular", "trata_de_la_empresa", "etiqueta"]])
if len(no_neutrales_ajenos):
    print("No tratan de la empresa, pero su etiqueta no es neutral:")
    display(no_neutrales_ajenos[["id", "titular", "trata_de_la_empresa", "etiqueta"]])


Etiquetados: 0 de 200
Reparto: {} · no tratan de la empresa: 0


## Comprobaciones

In [6]:
# Plantilla: 200 titulares distintos, ids del 1 al 200, como mucho 5 por empresa, y la misma muestra que se genera aquí
assert len(etiquetado) == TITULARES_A_MANO and etiquetado["id"].tolist() == list(range(1, TITULARES_A_MANO + 1))
assert etiquetado["titular"].is_unique
assert etiquetado["ticker"].value_counts().max() <= MAXIMO_POR_EMPRESA
assert etiquetado[["id", "ticker", "titular"]].equals(plantilla[["id", "ticker", "titular"]])
assert len(mal_escritos) == 0 and len(no_neutrales_ajenos) == 0
print("Plantilla: OK")

Plantilla: OK
